# FraudFinder - Environment Setup

## Overview

This series of labs are updated upon [FraudFinder](https://github.com/googlecloudplatform/fraudfinder) repository which builds a end-to-end real-time fraud detection system on Google Cloud. Throughout the FraudFinder labs, you will learn how to read historical bank transaction data stored in data warehouse, read from a live stream of new transactions, perform exploratory data analysis (EDA), do feature engineering, ingest features into a feature store, train a model using feature store, register your model in a model registry, evaluate your model, deploy your model to an endpoint, do real-time inference on your model with feature store, and monitor your model.

### Objective

In this notebook, you will setup your environment for Fraudfinder to be used in subsequent labs.

This lab uses the following Google Cloud services and resources:

- [Vertex AI](https://cloud.google.com/vertex-ai/)
- [BigQuery](https://cloud.google.com/bigquery/)
- [Google Cloud Storage](https://cloud.google.com/storage)
- [Pub/Sub](https://cloud.google.com/pubsub/)

Steps performed in this notebook:

- Setup your environment.
- Load historical bank transactions into BigQuery.
- Create Pub/Sub subscriptions to receive streaming data.
- Read data from BigQuery tables.
- Read data from Pub/Sub topics, which contain a live stream of new transactions.

### Setup your environment

Run the next cell to set your project ID and some of the other constants used in the lab.  

In [1]:
import random
import string
from typing import Union

import pandas as pd
from google.cloud import bigquery

# Generate unique ID to help w/ unique naming of certain pieces
ID = "".join(random.choices(string.ascii_lowercase + string.digits, k=5))

GCP_PROJECTS = !gcloud config get-value project
PROJECT_ID = GCP_PROJECTS[0]
BUCKET_NAME = f"{PROJECT_ID}-fraudfinder"
REGION = "us-central1"
TRAINING_DS_SIZE = 1000

### Create a Google Cloud Storage bucket and save the config data.

Next, we will create a Google Cloud Storage bucket and will save the config data in this bucket. After the cell operation finishes, you can navigate to [Google Cloud Storage](https://console.cloud.google.com/storage/) to see the GCS bucket. 

In [2]:
config = f"""
BUCKET_NAME          = \"{BUCKET_NAME}\"
PROJECT              = \"{PROJECT_ID}\"
REGION               = \"{REGION}\"
ID                   = \"{ID}\"
FEATURESTORE_ID      = \"fraudfinder_{ID}\"
MODEL_NAME           = \"ff_model\"
ENDPOINT_NAME        = \"ff_model_endpoint\"
TRAINING_DS_SIZE     = \"{TRAINING_DS_SIZE}\"
"""

!gsutil mb -l {REGION} gs://{BUCKET_NAME}

!echo '{config}' | gsutil cp - gs://{BUCKET_NAME}/config/notebook_env.py

Creating gs://qwiklabs-asl-02-f5c81170e4c5-fraudfinder/...
ServiceException: 409 A Cloud Storage bucket named 'qwiklabs-asl-02-f5c81170e4c5-fraudfinder' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.
Copying from <STDIN>...
/ [1 files][    0.0 B/    0.0 B]                                                
Operation completed over 1 objects.                                              


### Copy the historical transaction data into BigQuery tables

Now we will copy the historical transaction data and ingest it into BigQuery tables. For this, we will need to run `copy_bigquery_data.py`.

In [3]:
!python3 scripts/copy_bigquery_data.py $BUCKET_NAME

File copied from gs://cymbal-fraudfinder/datagen/hacked_customers_history.txt 
		 to gs://qwiklabs-asl-02-f5c81170e4c5-fraudfinder/datagen/hacked_customers_history.txt
File copied from gs://cymbal-fraudfinder/datagen/hacked_terminals_history.txt 
		 to gs://qwiklabs-asl-02-f5c81170e4c5-fraudfinder/datagen/hacked_terminals_history.txt
File copied from gs://cymbal-fraudfinder/datagen/demographics/customer_profiles.csv 
		 to gs://qwiklabs-asl-02-f5c81170e4c5-fraudfinder/datagen/demographics/customer_profiles.csv
File copied from gs://cymbal-fraudfinder/datagen/demographics/terminal_profiles.csv 
		 to gs://qwiklabs-asl-02-f5c81170e4c5-fraudfinder/datagen/demographics/terminal_profiles.csv
File copied from gs://cymbal-fraudfinder/datagen/demographics/customer_with_terminal_profiles.csv 
		 to gs://qwiklabs-asl-02-f5c81170e4c5-fraudfinder/datagen/demographics/customer_with_terminal_profiles.csv
BigQuery table created: `qwiklabs-asl-02-f5c81170e4c5`.tx.tx
BigQuery table created: `qwiklabs-a

### Check data in BigQuery

After ingesting our data into BigQuery, it's time to run some queries against the tables to inspect the data. You can also go to the [BigQuery console](https://console.cloud.google.com/bigquery) to see the data.

#### tx.tx
The `tx.tx` table contains the basic information about each transaction:
- `TX_ID` is a unique ID per transaction
- `TX_TS` is the timestamp of the transaction, in UTC
- `CUSTOMER_ID` is a unique 16-digit string ID per customer
- `TERMINAL_ID` is a unique 16-digit string ID per point-of-sale terminal
- `TX_AMOUNT` is the amount of money spent by the customer at a terminal, in dollars

In [4]:
%%bigquery
SELECT * FROM tx.tx LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

,TX_ID,TX_TS,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT
0,c830382a3d40d758582b9f85c793da807602ecf4,2024-06-08 08:23:17+00:00,7946629632082933,00064542,36.580000000
1,a58d8acf1e07624d995418fb77efbe3ef9902dc1,2024-06-08 19:34:30+00:00,5925748808382261,00064542,84.200000000
2,69fc4a3523ee4cc98d0076d4d993f5829c7bc998,2024-06-08 21:52:58+00:00,9942884307002811,00064542,67.940000000
3,bd02216f63c761e32595750b2abd2d39a0c6303a,2024-06-08 11:56:15+00:00,9147383035339177,00064542,18.500000000
4,0bd9949742c2fd86913a124c8bf588a5ab179b06,2024-06-08 15:24:22+00:00,9543432486781361,00064542,77.800000000


#### tx.txlabels
The `tx.txlabels` table contains information on whether each transation was fraud or not:
- `TX_ID` is a unique ID per transaction
- `TX_FRAUD` is 1 if the transaction was fraud, and 0 if the transaction was not fraudulent

In [5]:
%%bigquery
SELECT * FROM tx.txlabels LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

,TX_ID,TX_FRAUD
0,945f16523202b027998361745057dadd5c0fe8c6,0
1,2cdca1a892e6f2f105781e3d86e3b22bc84d1f19,0
2,13e47ca3d130af0aa3dfc6f0bd38bce6cacaf09c,0
3,615e41c559272e42b6eec5e233c8ccc0b9b24bba,0
4,1b71aa1997d4b521bb60d1d2329f60f12a38ba73,0


### Create Pub/Sub subscriptions for streaming transactions

Now, let's create Pub/Sub subscriptions to public Pub/Sub topics, where there is a constant flow of new transactions. This means you have, in your own Google Cloud project, subscriptions to the public Pub/Sub topics. You will receive a Pub/Sub message in your subscription every time a new transaction is streamed into the Pub/Sub topic.

There are two public Pub/Sub topics where there is a constant stream of live transactions occurring.

The following Pub/Sub topics are used for transactions:
```
projects/cymbal-fraudfinder/topics/ff-tx
projects/cymbal-fraudfinder/topics/ff-txlabels
```

In [6]:
%%bash
gcloud pubsub subscriptions create "ff-tx-sub" --topic="ff-tx" --topic-project="cymbal-fraudfinder"
gcloud pubsub subscriptions create "ff-tx-for-feat-eng-sub" --topic="ff-tx" --topic-project="cymbal-fraudfinder"
gcloud pubsub subscriptions create "ff-txlabels-sub" --topic="ff-txlabels" --topic-project="cymbal-fraudfinder"

ERROR: Failed to create subscription [projects/qwiklabs-asl-02-f5c81170e4c5/subscriptions/ff-tx-sub]: Resource already exists in the project (resource=ff-tx-sub).
ERROR: (gcloud.pubsub.subscriptions.create) Failed to create the following: [ff-tx-sub].
ERROR: Failed to create subscription [projects/qwiklabs-asl-02-f5c81170e4c5/subscriptions/ff-tx-for-feat-eng-sub]: Resource already exists in the project (resource=ff-tx-for-feat-eng-sub).
ERROR: (gcloud.pubsub.subscriptions.create) Failed to create the following: [ff-tx-for-feat-eng-sub].
ERROR: Failed to create subscription [projects/qwiklabs-asl-02-f5c81170e4c5/subscriptions/ff-txlabels-sub]: Resource already exists in the project (resource=ff-txlabels-sub).
ERROR: (gcloud.pubsub.subscriptions.create) Failed to create the following: [ff-txlabels-sub].


CalledProcessError: Command 'b'gcloud pubsub subscriptions create "ff-tx-sub" --topic="ff-tx" --topic-project="cymbal-fraudfinder"\ngcloud pubsub subscriptions create "ff-tx-for-feat-eng-sub" --topic="ff-tx" --topic-project="cymbal-fraudfinder"\ngcloud pubsub subscriptions create "ff-txlabels-sub" --topic="ff-txlabels" --topic-project="cymbal-fraudfinder"\n'' returned non-zero exit status 1.

### Reading messages from the Pub/Sub topics

In [ ]:
def read_from_sub(project_id, subscription_name, messages=10):
    """
    Read messages from a Pub/Sub subscription
    Args:
        project_id: project ID
        subscription_name: the name of a Pub/Sub subscription in your project
        messages: number of messages to read
    Returns:
        msg_data: list of messages in your Pub/Sub subscription as a Python dictionary
    """

    import ast

    from google.api_core import retry
    from google.cloud import pubsub_v1

    subscriber = pubsub_v1.SubscriberClient()
    subscription_path = subscriber.subscription_path(
        project_id, subscription_name
    )

    # Wrap the subscriber in a 'with' block to automatically call close() to
    # close the underlying gRPC channel when done.
    with subscriber:
        # The subscriber pulls a specific number of messages. The actual
        # number of messages pulled may be smaller than max_messages.
        response = subscriber.pull(
            subscription=subscription_path,
            max_messages=messages,
            retry=retry.Retry(deadline=300),
        )

        if len(response.received_messages) == 0:
            print("no messages")
            return

        ack_ids = []
        msg_data = []
        for received_message in response.received_messages:
            msg = ast.literal_eval(
                received_message.message.data.decode("utf-8")
            )
            msg_data.append(msg)
            ack_ids.append(received_message.ack_id)

        # Acknowledges the received messages so they will not be sent again.
        subscriber.acknowledge(subscription=subscription_path, ack_ids=ack_ids)

        print(
            f"Received and acknowledged {len(response.received_messages)} messages from {subscription_path}."
        )

        return msg_data

#### Reading from the `ff-tx-sub` subscription

Now let's read from the `ff-tx-sub` subscription. You should see some recent transactions (in UTC timezone).

In [ ]:
messages_tx = read_from_sub(
    project_id=PROJECT_ID, subscription_name="ff-tx-sub", messages=2
)

messages_tx

#### Reading from the `ff-txlabels-sub` subscription

We will do the same with `ff-txlabels-sub` subscription, which receives the same stream of transactions as `ff-tx-sub`, but also contain the ground-truth label, `TX_FRAUD`, if the transaction is fraudulent (1) or not (0).

In [ ]:
messages_txlabels = read_from_sub(
    project_id=PROJECT_ID, subscription_name="ff-txlabels-sub", messages=2
)

messages_txlabels

### END

Now you can go to the next notebook `01_exploratory_data_analysis.ipynb`

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.